# CRISP-DM Phase 3: Data Preparation (Hospital Regression Dataset)

In this phase, we tackle extreme dataset skew and complex string matrices. Our core objectives:
1. **Mathematical Junk Scrubbing**: Handle repeating header rows.
2. **NLP Text Vectorization**: The diseases are typed as complex comma-separated paragraphs. We will use Natural Language Processing to extract keyword importance.
3. **Logarithmic Scaling**: We have extreme 960g dose outliers. A log transformation will compress these mathematical extremes safely.
4. **Export Engine**: Generate final ML-ready `.csv` files.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

## 1. Load & Clean Raw Data
**Why are we doing this?** The raw CSV from the hospital database injected the column headers over and over again randomly as "rows". If we don't clear these string combinations out instantly, Python treats the numbers as text and crashes.

In [2]:
DATA_FILE = 'Hopsital Dataset.csv'
df_raw = pd.read_csv(DATA_FILE)

# 1. Remove Junk Header Rows
checks = [
    (df_raw['Age'].astype(str).str.strip() == 'Age'),
    (df_raw['Duration (days)'].astype(str).str.strip() == 'Duration (days)'),
    (df_raw['Gender'].astype(str).str.strip() == 'Gender')
]
mask = pd.Series(False, index=df_raw.index)
for c in checks: 
    mask = mask | c
df_clean = df_raw.loc[~mask].copy()

# 2. Force convert types
df_clean['Age_num'] = pd.to_numeric(df_clean['Age'], errors='coerce')
df_clean['Dosage_num'] = pd.to_numeric(df_clean['Dosage (gram)'], errors='coerce')
df_clean['Duration_num'] = pd.to_numeric(df_clean['Duration (days)'], errors='coerce')

# 3. Drop rows that failed conversion
df_clean.dropna(subset=['Age_num', 'Dosage_num', 'Duration_num'], inplace=True)
print(f"Clean Data Ready. Total Rows: {len(df_clean)}")

Clean Data Ready. Total Rows: 831


## 2. Train / Test Split
**Why are we doing this FIRST?** We must split the dataset into 80% Training and 20% Testing **BEFORE** we run our NLP algorithms. If we do it after, the model "memorizes" vocabulary from the test set, creating Data Leakage.

In [3]:
X = df_clean.drop(['Duration_num', 'Duration (days)', 'Age', 'Dosage (gram)'], axis=1)
y = df_clean['Duration_num']

# Note: NOT stratifying because target is continuous (Regression), not discrete.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set: {X_train.shape}")

Training set: (664, 9)


## 3. NLP Text Vectorization (TF-IDF)
**Why use TF-IDF instead of One-Hot Encoding?**
The `Diagnosis` column literally contains paragraphs (e.g. `heart failure, hcv, col`). If we used One-Hot, it would create 263 useless columns. 
Instead, Natural Language Processing (`TfidfVectorizer`) scans the sentences, finds the 15 most frequent/important words across all patients (like 'heart', 'cancer'), and creates numerical columns scoring how heavily related the patient is to that word!

In [4]:
text_cols = ['Diagnosis', 'Name of Drug', 'Indication']
tfidf_features_train = []
tfidf_features_test = []
feature_names_out = []

for col in text_cols:
    # Initialize NLP Vectorizer targeting the top 15 words per column
    vectorizer = TfidfVectorizer(max_features=15, stop_words='english')
    
    # Fit strictly on train data to prevent leakage
    train_mat = vectorizer.fit_transform(X_train[col].fillna(''))
    test_mat = vectorizer.transform(X_test[col].fillna(''))
    
    tfidf_features_train.append(pd.DataFrame(train_mat.toarray(), index=X_train.index))
    tfidf_features_test.append(pd.DataFrame(test_mat.toarray(), index=X_test.index))
    
    # Track column names
    feature_names_out.extend([f"{col}_{word}" for word in vectorizer.get_feature_names_out()])

# Combine all NLP features
df_tfidf_train = pd.concat(tfidf_features_train, axis=1)
df_tfidf_test = pd.concat(tfidf_features_test, axis=1)

df_tfidf_train.columns = feature_names_out
df_tfidf_test.columns = feature_names_out
print("NLP Processing Complete. Top words extracted!")
display(df_tfidf_train.head(3))

NLP Processing Complete. Top words extracted!


,Diagnosis_acute,Diagnosis_ccf,Diagnosis_chest,Diagnosis_ckd,Diagnosis_col,Diagnosis_copd,Diagnosis_dm,Diagnosis_hypertension,Diagnosis_ihd,Diagnosis_infection,...,Indication_dm,Indication_fever,Indication_infection,Indication_koch,Indication_left,Indication_lung,Indication_prevention,Indication_snake,Indication_type,Indication_uti
477,0.0,0.0,0.0,0.0,0.456703,0.0,0.0,0.537946,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
346,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
462,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Addressing Skew: Log Transformations
**Why `np.log1p`?** Our `Dosage` numbers mostly hover around 1-5g, but some spike to 960g! If we feed 960g to a Regression algorithm, the model line will completely break. A Logarithmic transformation cleanly compresses massive numbers into a smooth curve without deleting the data.
**Why `MinMaxScaler`?** We use this to scale `Age` cleanly between 0 and 1.

In [5]:
# 1. Log Transform Dosage
X_train_log = np.log1p(X_train['Dosage_num'])
X_test_log = np.log1p(X_test['Dosage_num'])

# 2. MinMax Scale Age
scaler = MinMaxScaler()
X_train_age = scaler.fit_transform(X_train[['Age_num']])
X_test_age = scaler.transform(X_test[['Age_num']])

# Reconstruct numeric dataframe
df_num_train = pd.DataFrame({'Log_Dosage': X_train_log, 'Scaled_Age': X_train_age.flatten()}, index=X_train.index)
df_num_test = pd.DataFrame({'Log_Dosage': X_test_log, 'Scaled_Age': X_test_age.flatten()}, index=X_test.index)
print("Numerical Skew mathematically stabilized.")

Numerical Skew mathematically stabilized.


## 5. Standard Categorical Processing
**Why One-Hot?** For simple text columns with low limits (`Gender`, `Route`, `Frequency`), standard binary flagging (0 or 1) is flawless.

In [6]:
cat_cols = ['Gender', 'Route', 'Frequency']

# handle_unknown='ignore' protects us if the test set has a category the train set missed
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Fit on Train, transform both
cat_train_mat = ohe.fit_transform(X_train[cat_cols].fillna('Missing'))
cat_test_mat = ohe.transform(X_test[cat_cols].fillna('Missing'))

cat_feature_names = ohe.get_feature_names_out(cat_cols)

df_cat_train = pd.DataFrame(cat_train_mat, columns=cat_feature_names, index=X_train.index)
df_cat_test = pd.DataFrame(cat_test_mat, columns=cat_feature_names, index=X_test.index)

## 6. Matrix Alignment and Export
We glue our NLP matrix, Log columns, and One-hot matrices together into the final perfect dataset.

In [7]:
FINAL_X_train = pd.concat([df_num_train, df_cat_train, df_tfidf_train], axis=1)
FINAL_X_test = pd.concat([df_num_test, df_cat_test, df_tfidf_test], axis=1)

print(f"Final Train Data Shape: {FINAL_X_train.shape}")

out_dir = os.path.join('data', 'processed', 'hospital')
os.makedirs(out_dir, exist_ok=True)

FINAL_X_train.to_csv(os.path.join(out_dir, 'X_train.csv'), index=False)
FINAL_X_test.to_csv(os.path.join(out_dir, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(out_dir, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(out_dir, 'y_test.csv'), index=False)

print(f"Success! Exported completely cleaned arrays to: {os.path.abspath(out_dir)}")

Final Train Data Shape: (664, 53)
Success! Exported completely cleaned arrays to: C:\Users\rahma\Desktop\machine learning project\regression_hospital_data _set\data\processed\hospital
